# 9. The Quay Crane Scheduling Problem
## Tier 3 — Advanced Algorithm (Genetic Algorithm)

Tier 2 gave us a fast heuristic, but it only explores **one** schedule. Tier 3 introduces a **metaheuristic**: a Genetic Algorithm (GA) that evolves a population of candidate schedules. By combining good solutions (crossover) and occasionally making small changes (mutation), the GA can find better makespans than simple greedy rules.

### Learning goals
- Understand the **GA components** for QCSP: chromosome representation, fitness, crossover, mutation.
- See how a population of schedules can be **evolved** over generations.
- Learn how to balance **exploration** (mutation) vs **exploitation** (crossover/selection).
- Compare GA performance against the Tier 2 heuristic baseline.

### What this notebook outputs
- A best-found schedule for a 10-bay, 3-crane instance using GA.
- A **convergence plot** (best makespan vs generation).
- A table of the final schedule and a Gantt visualization.
- A **performance comparison** with Tier 2 heuristic (percentage improvement).
- A short **what-if experiment** varying mutation rate to show its effect on convergence.

### Why the instance is larger
We use 10 bays and 3 cranes to give the GA enough search space to demonstrate its power, while still keeping runtimes reasonable for a teaching notebook.

In [ ]:
# Environment check (no installs here)
#
# Best practice: dependencies are preinstalled in the Docker/JupyterHub image.
# If you are running locally, install packages once in your environment.

try:
    import random
    import itertools
    from dataclasses import dataclass
    from typing import List, Dict, Tuple

    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
except ImportError as e:
    raise ImportError(
        "Missing dependency. Install: numpy, pandas, matplotlib. "
        "If you use the provided JupyterHub Docker image, these should already be installed."
    ) from e

print("Dependencies imported successfully.")

## Concrete QCSP instance (10 bays, 3 cranes)

We use a slightly larger instance than Tier 2:

- Bays: 1..10
- Processing times (minutes): [45, 30, 25, 35, 40, 20, 55, 28, 32, 38]
- Travel time between adjacent bays: 5 minutes
- Cranes: 3 (crane 0, 1, 2)

### GA chromosome representation
We encode a schedule as a **list of 10 integers**, each in {0,1,2}, indicating which crane serves each bay. This is simple and allows standard crossover/mutation operators.

### GA components
- **Fitness**: negative makespan (smaller makespan → higher fitness).
- **Selection**: tournament selection (pick 2 random individuals, keep the better).
- **Crossover**: one-point crossover (swap tails after a random cut point).
- **Mutation**: with probability `mut_rate`, change a random gene to another random crane.

In [ ]:
# ----------------------------
# Data model for the 10-bay, 3-crane instance
# ----------------------------
@dataclass(frozen=True)
class Bay:
    id: int
    position: int
    process_time: int

# Define the 10 bays
bays: List[Bay] = [
    Bay(id=1, position=1, process_time=45),
    Bay(id=2, position=2, process_time=30),
    Bay(id=3, position=3, process_time=25),
    Bay(id=4, position=4, process_time=35),
    Bay(id=5, position=5, process_time=40),
    Bay(id=6, position=6, process_time=20),
    Bay(id=7, position=7, process_time=55),
    Bay(id=8, position=8, process_time=28),
    Bay(id=9, position=9, process_time=32),
    Bay(id=10, position=10, process_time=38),
]

NUM_CRANES = 3
CRANE_IDS = list(range(NUM_CRANES))
TRAVEL_PER_BAY = 5.0

bay_by_id: Dict[int, Bay] = {b.id: b for b in bays}
bay_ids: List[int] = [b.id for b in bays]
positions = {b.id: b.position for b in bays}
process_times = {b.id: b.process_time for b in bays}

bays, positions, process_times

## Step 1 — GA helper functions (fitness, evaluation, crossover, mutation)

We will implement:
- `makespan_for_chromosome(chromosome)`: compute makespan for a given assignment.
- `fitness(chromosome)`: negative makespan (higher is better).
- `tournament_selection(pop, fitnesses, k=2)`: select one parent via tournament.
- `one_point_crossover(parent1, parent2)`: produce two children.
- `mutate(chromosome, mut_rate)`: apply mutation.

In [ ]:
def travel_time(pos_from: int, pos_to: int) -> float:
    """Travel time (minutes) as distance along the quay."""
    return TRAVEL_PER_BAY * abs(pos_from - pos_to)


def makespan_for_chromosome(chromosome: List[int]) -> float:
    """Compute makespan for a chromosome (list of crane assignments per bay)."""
    # Build crane sequences (bays per crane)
    crane_sequences: Dict[int, List[int]] = {cid: [] for cid in CRANE_IDS}
    for bay_id, cid in zip(bay_ids, chromosome):
        crane_sequences[cid].append(bay_id)
    # Sort each crane's bays by position to reduce travel
    for cid in CRANE_IDS:
        crane_sequences[cid].sort(key=lambda b: positions[b])
    # Compute completion time for each crane
    def crane_completion(seq: List[int]) -> float:
        if not seq:
            return 0.0
        time = 0.0
        current_pos = positions[seq[0]]
        for bid in seq:
            time += travel_time(current_pos, positions[bid])
            time += process_times[bid]
            current_pos = positions[bid]
        return time
    makespan = max(crane_completion(crane_sequences[cid]) for cid in CRANE_IDS)
    return makespan


def fitness(chromosome: List[int]) -> float:
    """Fitness is negative makespan (higher fitness = better)."""
    return -makespan_for_chromosome(chromosome)


def tournament_selection(pop: List[List[int]], fitnesses: List[float], k: int = 2) -> List[int]:
    """Select one parent via tournament selection (pick k random, keep best)."""
    indices = random.sample(range(len(pop)), k)
    best_idx = max(indices, key=lambda i: fitnesses[i])
    return pop[best_idx].copy()  # return a copy to avoid mutation side effects


def one_point_crossover(p1: List[int], p2: List[int]) -> Tuple[List[int], List[int]]:
    """One-point crossover: choose a cut point and swap tails."""
    if len(p1) != len(p2):
        raise ValueError("Parents must have same length")
    n = len(p1)
    if n <= 1:
        return p1.copy(), p2.copy()
    cut = random.randint(1, n - 1)
    child1 = p1[:cut] + p2[cut:]
    child2 = p2[:cut] + p1[cut:]
    return child1, child2


def mutate(chromosome: List[int], mut_rate: float) -> List[int]:
    """Apply mutation: with probability mut_rate, change a gene to another random crane."""
    mutated = chromosome.copy()
    for i in range(len(mutated)):
        if random.random() < mut_rate:
            # Choose a different crane (avoid staying the same)
            choices = [c for c in CRANE_IDS if c != mutated[i]]
            mutated[i] = random.choice(choices)
    return mutated


# Quick sanity checks
sample_chromosome = [random.choice(CRANE_IDS) for _ in bay_ids]
sample_makespan = makespan_for_chromosome(sample_chromosome)
sample_fitness = fitness(sample_chromosome)
print("Sample chromosome:", sample_chromosome)
print("Makespan (minutes):", sample_makespan)
print("Fitness:", sample_fitness)

## Step 2 — GA main loop

We will run the GA for a fixed number of generations, keeping track of:
- Best makespan per generation.
- Average makespan per generation.
- The final best chromosome (schedule).

In [ ]:
def run_ga(
    pop_size: int = 50,
    generations: int = 200,
    mut_rate: float = 0.05,
    elite_size: int = 2,
    seed: int = 42,
) -> Dict:
    """Run the Genetic Algorithm and return results."""
    random.seed(seed)
    np.random.seed(seed)

    # Initialize random population
    population = [
        [random.choice(CRANE_IDS) for _ in bay_ids] for _ in range(pop_size)
    ]

    # Track best makespan per generation
    best_makespan_per_gen = []
    avg_makespan_per_gen = []
    best_chromosome = None
    best_makespan = float("inf")

    for gen in range(generations):
        # Evaluate fitness
        fitnesses = [fitness(ind) for ind in population]
        makespans = [-f for f in fitnesses]

        # Record statistics
        gen_best = min(makespans)
        gen_avg = sum(makespans) / len(makespans)
        best_makespan_per_gen.append(gen_best)
        avg_makespan_per_gen.append(gen_avg)

        # Update global best
        if gen_best < best_makespan:
            best_makespan = gen_best
            best_idx = makespans.index(gen_best)
            best_chromosome = population[best_idx].copy()

        # Selection: create next generation
        next_pop = []

        # Elitism: keep top elite_size individuals
        sorted_indices = sorted(range(len(population)), key=lambda i: fitnesses[i], reverse=True)
        for i in range(elite_size):
            next_pop.append(population[sorted_indices[i]].copy())

        # Fill the rest via crossover + mutation
        while len(next_pop) < pop_size:
            parent1 = tournament_selection(population, fitnesses, k=3)
            parent2 = tournament_selection(population, fitnesses, k=3)
            child1, child2 = one_point_crossover(parent1, parent2)
            child1 = mutate(child1, mut_rate)
            child2 = mutate(child2, mut_rate)
            next_pop.append(child1)
            if len(next_pop) < pop_size:
                next_pop.append(child2)

        population = next_pop

    return {
        "best_chromosome": best_chromosome,
        "best_makespan": best_makespan,
        "best_makespan_per_gen": best_makespan_per_gen,
        "avg_makespan_per_gen": avg_makespan_per_gen,
    }


ga_result = run_ga(pop_size=50, generations=200, mut_rate=0.05, elite_size=2, seed=42)
print(f"GA best makespan (minutes): {ga_result['best_makespan']:.1f}")
print("Best chromosome (crane per bay):", ga_result["best_chromosome"])

## Step 3 — Convergence visualization

Plot the best and average makespan per generation to see how the GA improves over time.

In [ ]:
plt.figure(figsize=(7, 4))
gens = list(range(len(ga_result["best_makespan_per_gen"])))
plt.plot(gens, ga_result["best_makespan_per_gen"], label="Best makespan", color="#2E90FA")
plt.plot(gens, ga_result["avg_makespan_per_gen"], label="Average makespan", color="#12B76A", alpha=0.7)
plt.xlabel("Generation")
plt.ylabel("Makespan (minutes)")
plt.title("GA convergence for QCSP (10 bays, 3 cranes)")
plt.legend()
plt.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()

## Step 4 — Convert best chromosome to a readable schedule

We will transform the best chromosome into a table similar to Tier 1 and Tier 2, and plot a Gantt chart.

In [ ]:
def chromosome_to_schedule(chromosome: List[int]) -> pd.DataFrame:
    """Convert a chromosome into a flat schedule DataFrame (like Tier 1/2)."""
    # Build crane sequences
    crane_sequences: Dict[int, List[int]] = {cid: [] for cid in CRANE_IDS}
    for bay_id, cid in zip(bay_ids, chromosome):
        crane_sequences[cid].append(bay_id)
    for cid in CRANE_IDS:
        crane_sequences[cid].sort(key=lambda b: positions[b])

    rows = []
    for crane_id, seq in crane_sequences.items():
        time = 0.0
        current_pos = positions[seq[0]] if seq else None
        for step_idx, bay_id in enumerate(seq):
            bay_pos = positions[bay_id]
            if current_pos is None:
                continue
            travel = travel_time(current_pos, bay_pos)
            time += travel
            start = time
            time += process_times[bay_id]
            finish = time
            rows.append(
                {
                    "crane_id": crane_id,
                    "step": step_idx + 1,
                    "bay_id": bay_id,
                    "bay_position": bay_pos,
                    "process_time": process_times[bay_id],
                    "travel_time": travel,
                    "start": start,
                    "finish": finish,
                }
            )
            current_pos = bay_pos

    df = pd.DataFrame(rows).sort_values(["crane_id", "start"]).reset_index(drop=True)
    return df


ga_schedule_df = chromosome_to_schedule(ga_result["best_chromosome"])
ga_schedule_df

In [ ]:
# Verify makespan from the table
crane_completion_from_table = (
    ga_schedule_df.groupby("crane_id")["finish"].max().to_dict()
    if not ga_schedule_df.empty
    else {}
)
recomputed_makespan = max(crane_completion_from_table.values()) if crane_completion_from_table else 0.0
print("Per-crane completion times (minutes):", crane_completion_from_table)
print(f"Recomputed makespan (minutes): {recomputed_makespan:.1f}")
print("Matches GA result:", abs(recomputed_makespan - ga_result["best_makespan"]) < 1e-6)

## Step 5 — Gantt visualization of the GA schedule

In [ ]:
def plot_gantt(schedule: pd.DataFrame, title: str = "") -> None:
    """Plot a simple Gantt-style chart for a QCSP schedule."""
    if schedule.empty:
        print("No schedule to plot.")
        return

    fig, ax = plt.subplots(figsize=(8, 3.5))

    colors = {0: "#2E90FA", 1: "#12B76A", 2: "#F97316"}
    yticks = []
    yticklabels = []

    for crane_id in sorted(schedule["crane_id"].unique()):
        crane_rows = schedule[schedule["crane_id"] == crane_id]
        y = float(crane_id)
        yticks.append(y)
        yticklabels.append(f"Crane {crane_id}")

        for _, row in crane_rows.iterrows():
            start = row["start"]
            finish = row["finish"]
            duration = finish - start
            bay_id = int(row["bay_id"])

            ax.barh(
                y=y,
                width=duration,
                left=start,
                height=0.4,
                color=colors.get(crane_id, "#667085"),
                edgecolor="#101828",
                alpha=0.85,
            )
            ax.text(
                start + duration / 2.0,
                y,
                f"{bay_id}",
                ha="center",
                va="center",
                fontsize=9,
                color="white",
            )

    ax.set_yticks(yticks)
    ax.set_yticklabels(yticklabels)
    ax.set_xlabel("Time (minutes)")
    ax.set_title(title or "Tier 3 QCSP — GA schedule")
    ax.grid(True, axis="x", alpha=0.25)
    plt.tight_layout()
    plt.show()


plot_gantt(ga_schedule_df, title="Tier 3 QCSP — Genetic Algorithm schedule")

## Step 6 — Comparison with Tier 2 heuristic

We run the Tier 2 Enhanced LPT heuristic on the same 10-bay instance and compare makespans.

In [ ]:
# ---- Tier 2 heuristic on the same 10-bay instance ----
def enhanced_lpt_heuristic_10bay() -> Dict:
    sorted_bays = sorted(bay_ids, key=lambda b: process_times[b], reverse=True)
    crane_assignments: Dict[int, List[int]] = {cid: [] for cid in CRANE_IDS}
    crane_completion: Dict[int, float] = {cid: 0.0 for cid in CRANE_IDS}
    for bay_id in sorted_bays:
        best_crane = min(CRANE_IDS, key=lambda cid: crane_completion[cid])
        crane_assignments[best_crane].append(bay_id)
        # recompute completion (unsorted)
        time = 0.0
        current = positions[crane_assignments[best_crane][0]] if crane_assignments[best_crane] else None
        for b in crane_assignments[best_crane]:
            if current is None:
                continue
            time += travel_time(current, positions[b])
            time += process_times[b]
            current = positions[b]
        crane_completion[best_crane] = time
    # sort by position and recompute
    for cid in CRANE_IDS:
        crane_assignments[cid].sort(key=lambda b: positions[b])
        time = 0.0
        current = positions[crane_assignments[cid][0]] if crane_assignments[cid] else None
        for b in crane_assignments[cid]:
            if current is None:
                continue
            time += travel_time(current, positions[b])
            time += process_times[b]
            current = positions[b]
        crane_completion[cid] = time
    makespan = max(crane_completion[cid] for cid in CRANE_IDS)
    return {"makespan": makespan, "assignments": crane_assignments}


heuristic_10bay = enhanced_lpt_heuristic_10bay()
print("Tier 2 heuristic makespan (minutes):", heuristic_10bay["makespan"])
print("Tier 3 GA makespan (minutes):", ga_result["best_makespan"])

improvement_pct = ((heuristic_10bay["makespan"] - ga_result["best_makespan"]) / heuristic_10bay["makespan"]) * 100
print(f"Improvement over Tier 2 heuristic: {improvement_pct:.1f}%")

## Step 7 — Sensitivity: effect of mutation rate

We run the GA with different mutation rates (0.01, 0.05, 0.1) to see how exploration vs exploitation affects convergence.

In [ ]:
def run_ga_with_mut_rate(mut_rate: float, seed: int = 42) -> Dict:
    """Run GA with a specific mutation rate and return final best makespan."""
    result = run_ga(pop_size=50, generations=200, mut_rate=mut_rate, elite_size=2, seed=seed)
    return {"mut_rate": mut_rate, "best_makespan": result["best_makespan"]}


mut_rates = [0.01, 0.05, 0.1]
rows = [run_ga_with_mut_rate(mr, seed=42) for mr in mut_rates]
mut_sensitivity_df = pd.DataFrame(rows)
mut_sensitivity_df["best_makespan"] = mut_sensitivity_df["best_makespan"].round(1)
mut_sensitivity_df

In [ ]:
plt.figure(figsize=(6, 3.2))
plt.bar(
    [str(mr) for mr in mut_sensitivity_df["mut_rate"]],
    mut_sensitivity_df["best_makespan"],
    color="#2E90FA",
    edgecolor="#101828",
    alpha=0.85,
)
plt.xlabel("Mutation rate")
plt.ylabel("Best makespan (minutes)")
plt.title("Sensitivity of GA to mutation rate (10-bay, 3-crane instance)")
plt.grid(True, axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

## Step 8 — Why this Tier exists vs Tier 2

### Advantages vs Tier 2
- **Better solutions**: In our 10-bay example, the GA improved makespan by a few percent over the LPT heuristic.
- **Population-based search**: The GA explores many schedules simultaneously, reducing the risk of getting stuck in a poor local optimum.
- **Adaptability**: By tuning mutation/crossover, the GA can be adapted to different problem sizes and constraints.

### Disadvantages vs Tier 2
- **Computationally heavier**: The GA needs many fitness evaluations (pop_size × generations), while the heuristic runs in milliseconds.
- **More parameters**: The GA requires setting population size, generations, mutation rate, etc.
- **Stochastic variability**: Different random seeds can lead to different results; multiple runs may be needed for robustness.

### When to use this Tier
- When you have **moderate computational budget** (seconds to a few minutes) and want better quality than a simple heuristic.
- When the problem has **complex interactions** that greedy rules may miss (e.g., travel times, precedence constraints).
- When you need a **baseline** for even more advanced methods (RL, Digital Twin, Multi-Agent, Quantum).

### How this Tier connects to higher Tiers
- **Tier 4 (Reinforcement Learning)** will learn dispatching policies that adapt in real-time, going beyond the static evolutionary search of GA.
- **Tier 5+ (Digital Twin, Multi-Agent, Quantum)** will incorporate real-time data, system-of-systems coordination, and new computing paradigms.

For now, Tier 3 gives you a **metaheuristic that can beat simple heuristics** while still being understandable and tunable.